In [3]:
!which python

/opt/miniconda3/envs/bybit/bin/python


In [4]:
# %pip install --upgrade pip

In [5]:
# %pip uninstall pybit

In [6]:
# %pip install backtrader

In [7]:
# %pip install pandas matplotlib

In [1]:
from dotenv import load_dotenv
import os
import logging
import backtrader as bt
import pandas as pd
import matplotlib.pyplot as plt
import requests
import time
import hmac
import hashlib
from datetime import datetime, timedelta
import json
import threading
from typing import Dict, Optional, List

from pybit.unified_trading import HTTP

In [10]:
load_dotenv(override=True)

True

In [11]:
bybit_api_key = os.getenv("BYBIT_API_KEY")
bybit_secret_key = os.getenv('BYBIT_API_SECRET')


if bybit_api_key == None or bybit_secret_key == None:
    raise ValueError('API key not found')

bybit_api_key, bybit_secret_key

('public_key', 'secret_key_new')

In [4]:
session = HTTP(
    testnet=True,
    demo=True,
    api_key=bybit_api_key,
    api_secret=bybit_secret_key,
)

In [5]:
session.get_orderbook(category="linear", symbol="BTCUSDT")

{'retCode': 0,
 'retMsg': 'OK',
 'result': {'s': 'BTCUSDT',
  'b': [['116507.4', '0.001'],
   ['116507.2', '0.001'],
   ['116505.7', '0.002'],
   ['116504.8', '0.003'],
   ['116504.6', '0.001'],
   ['116504.3', '0.003'],
   ['116503.3', '0.002'],
   ['116503.1', '0.001'],
   ['116502.5', '0.002'],
   ['116501.6', '0.001'],
   ['116501.5', '0.001'],
   ['116501.1', '0.002'],
   ['116500.9', '0.002'],
   ['116499.5', '0.002'],
   ['116497.2', '0.002'],
   ['116496.8', '0.001'],
   ['116496.6', '0.002'],
   ['116494.9', '0.001'],
   ['116494.2', '0.002'],
   ['116492.9', '0.001'],
   ['116492.5', '0.002'],
   ['116490.2', '0.001'],
   ['116489.9', '0.003'],
   ['116489.7', '0.001'],
   ['116488.2', '0.001']],
  'a': [['116523.3', '0.001'],
   ['116533.3', '0.001'],
   ['116534', '0.001'],
   ['116540.2', '0.001'],
   ['116543.1', '0.002'],
   ['116545.7', '0.002'],
   ['116548.2', '0.002'],
   ['116550.7', '0.001'],
   ['116551.5', '0.001'],
   ['116557.2', '0.002'],
   ['116557.6', '0.00

In [6]:
session.get_account_info()

InvalidRequestError: API key is invalid. (ErrCode: 10003) (ErrTime: 18:46:07).
Request → GET https://api-demo-testnet.bybit.com/v5/account/info: .

In [ ]:
# data = session.place_order(
#     category="spot",
#     symbol="BTCUSDT",
#     side="Buy",
#     orderType="Limit",
#     qty="0.1",
#     price="15600",
#     timeInForce="PostOnly",
#     orderLinkId="spot-test-postonly",
#     isLeverage=0,
#     orderFilter="Order",
# )

# data['result']

In [ ]:
# session.cancel_order(
#     category="linear",
#     symbol="BTCPERP",
#     orderId="c6f055d9-7f21-4079-913d-e6523a9cfffa",
# )

In [6]:
params = {
            'accountType': 'UNIFIED',
            # coin="BTC",
        }

data = session.get_wallet_balance(**params)
data

{'retCode': 0,
 'retMsg': 'OK',
 'result': {'list': [{'totalEquity': '205906.760184',
    'accountIMRate': '0',
    'totalMarginBalance': '100000.2',
    'totalInitialMargin': '0',
    'accountType': 'UNIFIED',
    'totalAvailableBalance': '100000.2',
    'accountMMRate': '0',
    'totalPerpUPL': '0',
    'totalWalletBalance': '100000.2',
    'accountLTV': '0',
    'totalMaintenanceMargin': '0',
    'coin': [{'availableToBorrow': '',
      'bonus': '0',
      'accruedInterest': '0',
      'availableToWithdraw': '',
      'totalOrderIM': '0',
      'equity': '50000',
      'totalPositionMM': '0',
      'usdValue': '49993.7',
      'unrealisedPnl': '0',
      'collateralSwitch': True,
      'spotHedgingQty': '0',
      'borrowAmount': '0.000000000000000000',
      'totalPositionIM': '0',
      'walletBalance': '50000',
      'cumRealisedPnl': '0',
      'locked': '0',
      'marginCollateral': True,
      'coin': 'USDC'},
     {'availableToBorrow': '',
      'bonus': '0',
      'accruedI

In [7]:
params = {
            'category': 'spot',
            'symbol': 'BTCUSDT'
        }

data = session.get_tickers(**params)

data['result']['list']

[{'symbol': 'BTCUSDT',
  'bid1Price': '107411.3',
  'bid1Size': '0.00497',
  'ask1Price': '107411.7',
  'ask1Size': '0.00497',
  'lastPrice': '107405.63',
  'prevPrice24h': '106735.78',
  'price24hPcnt': '0.0063',
  'highPrice24h': '107472.2',
  'lowPrice24h': '106690.23',
  'turnover24h': '2960813.68817106',
  'volume24h': '27.628798',
  'usdIndexPrice': '103229.009476'}]

In [8]:
data = session.get_open_orders(
    category="linear",
    symbol="ETHUSDT",
    openOnly=0,
    limit=1,
)


data['result']['list']

[]

In [9]:
class ByBitAPI:
    """Класс для работы с API ByBit (торговые и рыночные данные)"""
    
    def __init__(self, api_key: str = os.getenv("BYBIT_API_KEY"), api_secret: str = os.getenv('BYBIT_API_SECRET'), testnet: bool = True):
        """
        Инициализация API клиента
        
        Args:
            api_key: API ключ
            api_secret: API секрет
            testnet: Использовать тестовую сеть
        """
        self.api_key = api_key
        self.api_secret = api_secret
        self.testnet = testnet
        
        if testnet:
            self.base_url = "https://api-testnet.bybit.com"
        else:
            self.base_url = "https://api.bybit.com"
            
        self.session = HTTP(testnet=True,
                            demo=True,
                            api_key=self.api_key,
                            api_secret=self.api_secret,
                        )
    
    def _generate_signature(self, params: dict, timestamp: str) -> str:
        """Генерация подписи для приватных запросов"""
        self.logger.debug('Call _generate_signature')
        if not self.api_secret:
            self.logger.debug('Secret key is not define')
            return ""
            
        # Создание строки параметров
        param_str = "&".join([f"{k}={v}" for k, v in sorted(params.items())])
        self.logger.debug(f'param_str: {param_str}')
        sign_str = f"{timestamp}{self.api_key}5000{param_str}"
        
        return hmac.new(
            self.api_secret.encode('utf-8'),
            sign_str.encode('utf-8'),
            hashlib.sha256
        ).hexdigest()
    
    def _make_request(self, method: str, endpoint: str, params: dict = None, private: bool = False):
        """Универсальный метод для выполнения запросов"""
        self.logger.debug('Call _make_request')
        if params is None:
            params = {}
        
        headers = {
            'Content-Type': 'application/json'
        }
        
        if private and self.api_key:
            timestamp = str(int(time.time() * 1000))
            headers.update({
                'X-BAPI-API-KEY': self.api_key,
                'X-BAPI-TIMESTAMP': timestamp,
                'X-BAPI-RECV-WINDOW': '5000',
                'X-BAPI-SIGN': self._generate_signature(params, timestamp)
            })
        
        try:
            if method.upper() == 'GET':
                
                response = requests.get(
                    self.base_url + endpoint,
                    params=params,
                    headers=headers
                )
            else:
                
                response = requests.post(
                    self.base_url + endpoint,
                    json=params,
                    headers=headers
                )
            
            response.raise_for_status()
            return response.json()
            
        except requests.RequestException as e:
            print(f"Ошибка запроса к API: {e}")
            return None
        except Exception as e:
            print(f"Ошибка обработки запроса: {e}")
            return None
    
    # РЫНОЧНЫЕ ДАННЫЕ
    def get_kline_data(self, symbol: str, limit: int = 1000, 
                      category: str = 'linear', interval: str = 'D') -> Optional[pd.DataFrame]:
        """Получение исторических данных свечей"""
        
        params = {
            'category': category,
            'symbol': symbol,
            'interval': interval,
            'limit': limit
        }
        
        data = self.session.get_kline(**params)

        # Преобразование в DataFrame
        klines = data['result']['list']

        df = pd.DataFrame(klines, columns=[
            'timestamp', 'open', 'high', 'low', 'close', 'volume', 'turnover'
        ])

        # Преобразование типов данных
        df['timestamp'] = pd.to_datetime(df['timestamp'].astype(int), unit='ms')
        for col in ['open', 'high', 'low', 'close', 'volume', 'turnover']:
            df[col] = df[col].astype(float)

        df.set_index('timestamp', inplace=True)
        df.sort_index(inplace=True)
        return df
        
    
    def get_ticker(self, symbol: str, category: str = 'spot') -> Optional[dict]:
        """Получение текущей цены и статистики по символу"""
        
        params = {
            'category': category,
            'symbol': symbol
        }

        data = self.session.get_tickers(**params)
        
        tickers = data['result']['list']
        
        return tickers[0] if tickers else None
        
    
    # ТОРГОВЫЕ ОПЕРАЦИИ
    def get_account_balance(self, accountType: str = 'UNIFIED') -> Optional[dict]:
        """Получение баланса аккаунта"""
        
        params = {
            'accountType': accountType
        }
        
        data = self.session.get_wallet_balance(**params)

        return data['result']['list']
    
    def get_positions(self, category: str = 'spot', symbol: str = None) -> Optional[List[dict]]:
        """Получение открытых позиций"""
        
        params = {
            'category': category
        }
        
        if symbol:
            params['symbol'] = symbol
        
        
        data = self.session.get_positions(**params)

        return data['result']['list']
    
    def place_order(self, symbol: str, side: str, orderType: str, qty: str, 
                   price: str = None, category: str = 'spot', 
                   timeInForce: str = 'GTC') -> Optional[dict]:
        """
        Размещение ордера
        
        Args:
            symbol: Торговая пара
            side: 'Buy' или 'Sell'
            orderType: 'Market' или 'Limit'
            qty: Количество
            price: Цена (для лимитных ордеров)
            category: Категория рынка
            timeInForce: Время действия ордера
        """
        
        params = {
            'category': category,
            'symbol': symbol,
            'side': side,
            'orderType': orderType,
            'qty': qty,
            'timeInForce': timeInForce
        }
        
        if price and orderType == 'Limit':
            params['price'] = price
        
        data = self.session.place_order(**params)
        
        return data['result']
    
    def get_open_orders(self, symbol: str = None, category: str = 'spot') -> Optional[List[dict]]:
        """Получение открытых ордеров"""
    #     endpoint = "/v5/order/realtime"
        
        params = {
            'category': category
        }
        
        if symbol:
            params['symbol'] = symbol
        
        data = self.session.get_open_orders(**params)

        return data['result']['list']

    def cancel_order(self, symbol: str, orderId: str = None, 
                    orderLinkId: str = None, category: str = 'spot') -> Optional[dict]:
        """Отмена ордера"""
    #     endpoint = "/v5/order/cancel"
        
        params = {
            'category': category,
            'symbol': symbol
        }
        
        if orderId:
            params['orderId'] = orderId
        elif orderLinkId:
            params['orderLinkId'] = orderLinkId
        else:
            print("Необходимо указать orderId или orderLinkId")
            return None
        
        data = self.session.cancel_order(**params)
        
        return data['result']

In [10]:
class LiveTradingStrategy:
    """Класс для торговли в реальном времени"""
    
    def __init__(self, api_key: str, api_secret: str, symbol: str = 'BTCUSDT',
                 category: str = 'spot', fast_ma: int = 20, slow_ma: int = 50,
                 trade_amount: float = 0.001, testnet: bool = True):
        """
        Инициализация торговой стратегии
        
        Args:
            api_key: API ключ ByBit
            api_secret: API секрет ByBit
            symbol: Торговая пара
            category: Категория рынка
            fast_ma: Период быстрой MA
            slow_ma: Период медленной MA
            trade_amount: Размер торговой позиции
            testnet: Использовать тестовую сеть
        """
        self.api = ByBitAPI(api_key, api_secret, testnet)
        self.symbol = symbol
        self.category = category
        self.fast_ma = fast_ma
        self.slow_ma = slow_ma
        self.trade_amount = trade_amount
        
        self.is_running = False
        self.current_position = None
        self.last_signal = None
        
        print(f"Инициализация торговой стратегии:")
        print(f"Символ: {symbol} ({category})")
        print(f"MA периоды: {fast_ma}/{slow_ma}")
        print(f"Размер позиции: {trade_amount}")
        print(f"Тестовая сеть: {testnet}")
    
    def get_market_data(self, limit: int = 100) -> Optional[pd.DataFrame]:
        """Получение рыночных данных"""
        return self.api.get_kline_data(
            symbol=self.symbol,
            # interval='1',  # 1 минута
            limit=limit,
            category=self.category
        )
    
    def calculate_signals(self, data: pd.DataFrame) -> Dict[str, any]:
        """Расчет торговых сигналов"""
        if len(data) < max(self.fast_ma, self.slow_ma):
            return {'signal': 'WAIT', 'reason': 'Недостаточно данных'}
        
        # Расчет скользящих средних
        data['fast_ma'] = data['close'].rolling(window=self.fast_ma).mean()
        data['slow_ma'] = data['close'].rolling(window=self.slow_ma).mean()
        
        # Получение последних значений
        current_fast_ma = data['fast_ma'].iloc[-1]
        current_slow_ma = data['slow_ma'].iloc[-1]
        prev_fast_ma = data['fast_ma'].iloc[-2]
        prev_slow_ma = data['slow_ma'].iloc[-2]
        
        current_price = data['close'].iloc[-1]
        
        # Определение сигнала
        signal = 'HOLD'
        reason = ''
        
        # Пересечение вверх - сигнал на покупку
        if (prev_fast_ma <= prev_slow_ma and current_fast_ma > current_slow_ma):
            signal = 'BUY'
            reason = f'Пересечение MA вверх: {current_fast_ma:.6f} > {current_slow_ma:.6f}'
        
        # Пересечение вниз - сигнал на продажу
        elif (prev_fast_ma >= prev_slow_ma and current_fast_ma < current_slow_ma):
            signal = 'SELL'
            reason = f'Пересечение MA вниз: {current_fast_ma:.6f} < {current_slow_ma:.6f}'
        
        return {
            'signal': signal,
            'reason': reason,
            'current_price': current_price,
            'fast_ma': current_fast_ma,
            'slow_ma': current_slow_ma,
            'timestamp': data.index[-1]
        }
    
    def get_account_status(self) -> Dict[str, any]:
        """Получение статуса аккаунта"""
        balance = self.api.get_account_balance()
        positions = self.api.get_positions(category=self.category, symbol=self.symbol)
        orders = self.api.get_open_orders(symbol=self.symbol, category=self.category)
        
        return {
            'balance': balance,
            'positions': positions,
            'open_orders': orders
        }
    
    def execute_trade(self, signal: str, current_price: float) -> bool:
        """Выполнение торговой операции"""
        try:
            if signal == 'BUY' and not self.current_position:
                # Покупка
                result = self.api.place_order(
                    symbol=self.symbol,
                    side='Buy',
                    orderType='Market',
                    qty=str(self.trade_amount),
                    category=self.category
                )
                
                if result:
                    self.current_position = 'LONG'
                    print(f"✅ ПОКУПКА ИСПОЛНЕНА: {self.trade_amount} {self.symbol} по цене ~{current_price:.6f}")
                    return True
                else:
                    print("❌ Ошибка выполнения покупки")
                    return False
            
            elif signal == 'SELL' and self.current_position == 'LONG':
                # Продажа
                result = self.api.place_order(
                    symbol=self.symbol,
                    side='Sell',
                    orderType='Market',
                    qty=str(self.trade_amount),
                    category=self.category
                )
                
                if result:
                    self.current_position = None
                    print(f"✅ ПРОДАЖА ИСПОЛНЕНА: {self.trade_amount} {self.symbol} по цене ~{current_price:.6f}")
                    return True
                else:
                    print("❌ Ошибка выполнения продажи")
                    return False
            
            return False
            
        except Exception as e:
            print(f"❌ Ошибка выполнения торговой операции: {e}")
            return False
    
    def analyze_and_trade(self):
        """Анализ рынка и выполнение торговых операций"""
        
        try:
            # Получение рыночных данных
            data = self.get_market_data(limit=max(self.fast_ma, self.slow_ma) + 10)
            
            if data is None or len(data) == 0:
                print("❌ Не удалось получить рыночные данные")
                return
            
            # Расчет сигналов
            analysis = self.calculate_signals(data)
            
            # Получение статуса аккаунта
            account_status = self.get_account_status()
            
            # Вывод информации
            print(f"\n{'='*60}")
            print(f"📊 АНАЛИЗ РЫНКА - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"{'='*60}")
            print(f"Символ: {self.symbol}")
            print(f"Текущая цена: {analysis['current_price']:.6f}")
            print(f"Быстрая MA ({self.fast_ma}): {analysis['fast_ma']:.6f}")
            print(f"Медленная MA ({self.slow_ma}): {analysis['slow_ma']:.6f}")
            print(f"Сигнал: {analysis['signal']}")
            print(f"Причина: {analysis['reason']}")
            print(f"Текущая позиция: {self.current_position or 'НЕТ'}")
            
            # Выполнение торговой операции
            
            if analysis['signal'] in ['BUY', 'SELL'] and analysis['signal'] != self.last_signal:
                print(f"\n🔔 НОВЫЙ ТОРГОВЫЙ СИГНАЛ: {analysis['signal']}")
                print(f'DEBUG: {analysis['signal'], analysis['current_price']}')
                if self.execute_trade(analysis['signal'], analysis['current_price']):
                    self.last_signal = analysis['signal']
                    
                    # Обновление статуса после сделки
                    time.sleep(2)  # Ожидание обновления данных
                    account_status = self.get_account_status()
            
            # Вывод статуса аккаунта
            if account_status['balance']:
                print(f"\n💰 СТАТУС АККАУНТА:")
                for i in range(len(account_status['balance'][0]['coin'])):
                    coin = account_status['balance'][0]['coin'][i]['coin']
                    balance = account_status['balance'][0]['coin'][i]['walletBalance']
                    print(f'{coin}: {balance}')
                
            if account_status['open_orders']:
                print(f"\n📋 ОТКРЫТЫЕ ОРДЕРА: {len(account_status['open_orders'])}")
                # for order in account_status['open_orders'][:3]:  # Показать первые 3
                #     print(f"  {order['side']} {order['qty']} {order['symbol']} @ {order.get('price', 'Market')}")
            
        except Exception as e:
            print(f"❌ Ошибка в analyze_and_trade: {e}")
    
    def start_live_trading(self, interval_seconds: int = 60):
        """Запуск торговли в реальном времени"""
        print(f"\n🚀 ЗАПУСК LIVE ТОРГОВЛИ")
        print(f"Интервал анализа: {interval_seconds} секунд")
        print(f"Для остановки нажмите Ctrl+C")
        
        self.is_running = True
        
        try:
            while self.is_running:
                self.analyze_and_trade()
                time.sleep(interval_seconds)
                
        except KeyboardInterrupt:
            print(f"\n🛑 ОСТАНОВКА ТОРГОВЛИ")
            self.is_running = False
        except Exception as e:
            print(f"❌ Критическая ошибка: {e}")
            self.is_running = False

In [11]:
class MACrossStrategy(bt.Strategy):
    """Стратегия пересечения скользящих средних для бэктестинга"""
    
    def __init__(self, fast_ma: int = 20, slow_ma: int = 50, printlog: bool = False):
        self.fast_ma = bt.indicators.SimpleMovingAverage(
            self.data.close, period=fast_ma)
        self.slow_ma = bt.indicators.SimpleMovingAverage(
            self.data.close, period=slow_ma)
        self.crossover = bt.indicators.CrossOver(self.fast_ma, self.slow_ma)
        self.order = None
        self.printlog = printlog
    
    def log(self, txt, dt=None):
        dt = dt or self.datas[0].datetime.date(0)
        if self.printlog:
            print(f'{dt.isoformat()}: {txt}')
    
    def notify_order(self, order):
        if order.status in [order.Completed]:
            if order.isbuy():
                self.log(f'ПОКУПКА: {order.executed.price:.6f}')
            else:
                self.log(f'ПРОДАЖА: {order.executed.price:.6f}')
        self.order = None
    
    def next(self):
        if self.order:
            return
        
        if not self.position:
            if self.crossover > 0:
                self.order = self.buy()
        else:
            if self.crossover < 0:
                self.order = self.sell()

In [12]:
def run_backtest(symbol='BTCUSDT', category='spot', interval='D', 
                fast_ma=20, slow_ma=50, initial_cash=10000, 
                commission=0.001, limit=500):
    """Функция для бэктестинга (упрощенная версия)"""
    
    # Создание API клиента для получения данных
    api = ByBitAPI()
    
    # Получение данных
    print(f"Загрузка данных {symbol}...")
    data = api.get_kline_data(symbol=symbol, interval=interval, 
                             limit=limit, category=category)
    
    if data is None or data.empty:
        print("Ошибка загрузки данных")
        return None
    
    # Настройка backtrader
    cerebro = bt.Cerebro()
    cerebro.addstrategy(MACrossStrategy, fast_ma=fast_ma, slow_ma=slow_ma)
    
    bt_data = bt.feeds.PandasData(dataname=data)
    cerebro.adddata(bt_data)
    
    cerebro.broker.setcash(initial_cash)
    cerebro.broker.setcommission(commission=commission)
    
    # Добавление анализаторов
    cerebro.addanalyzer(bt.analyzers.Returns, _name='returns')
    cerebro.addanalyzer(bt.analyzers.SharpeRatio, _name='sharpe')
    cerebro.addanalyzer(bt.analyzers.DrawDown, _name='drawdown')
    
    # Запуск
    results = cerebro.run()
    strategy = results[0]
    
    # Результаты
    final_value = cerebro.broker.getvalue()
    total_return = (final_value - initial_cash) / initial_cash * 100
    
    print(f"\n{'='*50}")
    print(f"РЕЗУЛЬТАТЫ БЭКТЕСТА")
    print(f"{'='*50}")
    print(f"Символ: {symbol}")
    print(f"MA: {fast_ma}/{slow_ma}")
    print(f"Начальный капитал: ${initial_cash:,.2f}")
    print(f"Конечный капитал: ${final_value:,.2f}")
    print(f"Доходность: {total_return:.2f}%")
    
    # График
    cerebro.plot(style='candlestick')
    plt.show()
    
    return {
        'total_return': total_return,
        'final_value': final_value
    }

In [13]:
# Параметры стратегии
SYMBOL = "BTCUSDT"
CATEGORY = "linear"  #linear, inverse
FAST_MA = 5   # Быстрая MA (уменьшено для быстрых сигналов)
SLOW_MA = 15  # Медленная MA
TRADE_AMOUNT = 0.001  # Размер позиции в BTC
TESTNET = True  # True для тестовой сети, False для основной

trader = LiveTradingStrategy(
    api_key=bybit_api_key,  # Без API ключей
    api_secret=bybit_secret_key,
    symbol=SYMBOL,
    category=CATEGORY,
    fast_ma=FAST_MA,
    slow_ma=SLOW_MA,
    trade_amount=TRADE_AMOUNT,
    testnet=TESTNET
)

# Однократный анализ
trader.analyze_and_trade()

Инициализация торговой стратегии:
Символ: BTCUSDT (linear)
MA периоды: 5/15
Размер позиции: 0.001
Тестовая сеть: True

📊 АНАЛИЗ РЫНКА - 2025-06-05 20:54:14
Символ: BTCUSDT
Текущая цена: 106500.000000
Быстрая MA (5): 105961.640000
Медленная MA (15): 103860.713333
Сигнал: HOLD
Причина: 
Текущая позиция: НЕТ

💰 СТАТУС АККАУНТА:
USDC: 50000
BTC: 1
ETH: 1
USDT: 50000


In [14]:
print(f"\n🔍 ЗАПУСК БЭКТЕСТИНГА")
results = run_backtest(
    symbol=SYMBOL,
    category=CATEGORY,
    fast_ma=FAST_MA,
    slow_ma=SLOW_MA,
    limit=200
)


🔍 ЗАПУСК БЭКТЕСТИНГА
Загрузка данных BTCUSDT...

РЕЗУЛЬТАТЫ БЭКТЕСТА
Символ: BTCUSDT
MA: 5/15
Начальный капитал: $10,000.00
Конечный капитал: $10,000.00
Доходность: 0.00%


<IPython.core.display.Javascript object>

In [15]:
results = run_backtest(
    symbol=SYMBOL,
    category=CATEGORY,
    fast_ma=15,
    slow_ma=50,
    limit=200
)

Загрузка данных BTCUSDT...

РЕЗУЛЬТАТЫ БЭКТЕСТА
Символ: BTCUSDT
MA: 15/50
Начальный капитал: $10,000.00
Конечный капитал: $10,000.00
Доходность: 0.00%


<IPython.core.display.Javascript object>

In [103]:
# if __name__ == '__main__':

print("🔧 НАСТРОЙКА ТОРГОВОЙ СТРАТЕГИИ")
print("="*50)

# Настройки API (ОБЯЗАТЕЛЬНО ЗАМЕНИТЕ НА СВОИ!)
API_KEY = "YOUR_API_KEY_HERE"
API_SECRET = "YOUR_API_SECRET_HERE"
TESTNET = True  # True для тестовой сети, False для основной

# Параметры стратегии
SYMBOL = "BTCUSDT"
CATEGORY = "linear"  #linear, inverse
FAST_MA = 5   # Быстрая MA (уменьшено для быстрых сигналов)
SLOW_MA = 15  # Медленная MA
TRADE_AMOUNT = 0.001  # Размер позиции в BTC

print("Выберите режим работы:")
print("1. Бэктестинг стратегии")
print("2. Live торговля (требуются API ключи)")
print("3. Анализ рынка без торговли")

choice = input("\nВведите номер (1-3): ").strip()

if choice == "1":
    # Бэктестинг
    print(f"\n🔍 ЗАПУСК БЭКТЕСТИНГА")
    results = run_backtest(
        symbol=SYMBOL,
        category=CATEGORY,
        fast_ma=FAST_MA,
        slow_ma=SLOW_MA,
        limit=200
    )

elif choice == "2":
    # Live торговля
    if API_KEY == "YOUR_API_KEY_HERE" or API_SECRET == "YOUR_API_SECRET_HERE":
        print("❌ ОШИБКА: Необходимо указать реальные API ключи!")
        print("Получите их на https://testnet.bybit.com (для тестовой сети)")
        print("или на https://www.bybit.com (для основной сети)")
    else:
        trader = LiveTradingStrategy(
            api_key=API_KEY,
            api_secret=API_SECRET,
            symbol=SYMBOL,
            category=CATEGORY,
            fast_ma=FAST_MA,
            slow_ma=SLOW_MA,
            trade_amount=TRADE_AMOUNT,
            testnet=TESTNET
        )
        
        # Запуск торговли
        trader.start_live_trading(interval_seconds=30)

elif choice == "3":
    # Только анализ
    print(f"\n📊 АНАЛИЗ РЫНКА БЕЗ ТОРГОВЛИ")
    
    trader = LiveTradingStrategy(
        api_key="",  # Без API ключей
        api_secret="",
        symbol=SYMBOL,
        category=CATEGORY,
        fast_ma=FAST_MA,
        slow_ma=SLOW_MA,
        trade_amount=TRADE_AMOUNT,
        testnet=TESTNET
    )
    
    # Однократный анализ
    trader.analyze_and_trade()

else:
    print("❌ Неверный выбор")

🔧 НАСТРОЙКА ТОРГОВОЙ СТРАТЕГИИ
Выберите режим работы:
1. Бэктестинг стратегии
2. Live торговля (требуются API ключи)
3. Анализ рынка без торговли

🔍 ЗАПУСК БЭКТЕСТИНГА
Загрузка данных BTCUSDT...

РЕЗУЛЬТАТЫ БЭКТЕСТА
Символ: BTCUSDT
MA: 5/15
Начальный капитал: $10,000.00
Конечный капитал: $10,000.00
Доходность: 0.00%


<IPython.core.display.Javascript object>